# Amazon Bedrock — Connection Setup

This notebook sets up credentials and a client for calling Claude models through Amazon Bedrock.

**Do not commit real credentials to this notebook.** Fill in the placeholders below via environment variables (recommended) or a local `.env` file that's excluded from version control.

In [1]:
# Install dependencies (uncomment as needed)
# %pip install boto3
# %pip install "anthropic[bedrock]"
# %pip install python-dotenv

## 1. Credentials (placeholders)

Bedrock auth is standard AWS credentials (access key / secret key / session token, or an assumed role) — not a separate "API key". Prefer environment variables or `~/.aws/credentials` over hardcoding values here.

In [2]:
import os
from dotenv import load_dotenv

# Loads AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_REGION from a local
# .env file (already created at the project root and covered by .gitignore).
# Put your NEW/rotated key pair there — never hardcode secrets in this notebook.
load_dotenv()

required = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing required env vars in .env: {missing}")

AWS_REGION = os.environ["AWS_REGION"]

## 2. Client setup — Option A: boto3 (native AWS SDK)

Use this if you want to call the raw Bedrock `InvokeModel` / `Converse` API directly.

In [3]:
import boto3

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    # boto3 picks up AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_SESSION_TOKEN
    # from the environment automatically — no need to pass them explicitly.
)

# Claude Haiku 4.5 on Bedrock requires the cross-region inference profile ID
# (the "us." prefix), not the bare "anthropic.claude-haiku-4-5-..." model ID.
BEDROCK_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

## 2. Client setup — Option B: Anthropic SDK's Bedrock (Mantle) client

Use this if you want the same `messages.create(...)` interface as the first-party Anthropic SDK, backed by Bedrock. Requires `pip install "anthropic[bedrock]"`.

> **Note:** on this account, the Mantle client only recognized the bare `anthropic.claude-haiku-4-5` ID (not the `us.`-prefixed inference profile ID boto3's Converse API needed), and that bare ID isn't enabled for this account/region. Option A (boto3) is the one confirmed working below — keeping this cell for reference/future use.

In [4]:
from anthropic import AnthropicBedrockMantle

anthropic_bedrock_client = AnthropicBedrockMantle(
    aws_region=AWS_REGION,
    # Credentials are resolved the same way boto3 resolves them (env vars,
    # ~/.aws/credentials, instance role, etc.) — nothing else to pass here.
)

## 3. Test the connection

Run whichever option you set up above.

In [5]:
# Option A test (boto3 Converse API) — confirmed working with Claude Haiku 4.5
""" test_prompt = (
    "Explain what Amazon Bedrock is, and describe how it works with Claude "
    "models — including how requests are routed, how authentication differs "
    "from the first-party Anthropic API, and what model IDs look like on "
    "Bedrock."
)"""
test_prompt = ("Please give me list of 5 best practices for using Amazon Bedrock with Claude models."
)

response = bedrock_runtime.converse(
    modelId=BEDROCK_MODEL_ID,
    messages=[{"role": "user", "content": [{"text": test_prompt}]}],
    inferenceConfig={"maxTokens": 512},
)
print(response["output"]["message"]["content"][0]["text"])

# 5 Best Practices for Using Amazon Bedrock with Claude Models

## 1. **Use System Prompts Effectively**
- Define clear system prompts to establish Claude's behavior, tone, and constraints
- System prompts are more reliable than embedding instructions in user messages
- Helps maintain consistent responses across multiple requests

## 2. **Implement Prompt Caching**
- Use prompt caching (available for Claude 3.5 Sonnet and later) to reduce latency and costs
- Cache large context windows, documents, or frequently used prompts
- Ideal for multi-turn conversations or repetitive tasks with similar context

## 3. **Optimize Token Usage**
- Monitor input and output token consumption to manage costs
- Use appropriate model sizes (Haiku for simple tasks, Opus for complex reasoning)
- Truncate or summarize large documents rather than including full text unnecessarily

## 4. **Structure Requests with Clear Formatting**
- Use XML tags or markdown for organizing complex prompts
- Include explicit e